# Task 2 — Generate Production Code and Design Optimized Test Cases

This notebook is the first part of Task 2. It reads the acceptance tests produced in Task 1 and uses a local LLM (Ollama / llama3) to generate a single, coherent `TrafficLight` Python class whose methods implement the behaviour described by those ATs.

**Why one class in one prompt?** The methods share state (current colours, mode, timers) and call each other (`execute_cycle` calls `yellow_transition`, etc.). Generating them separately would make the model invent inconsistent assumptions each time, producing pieces that don't fit together.

**Pipeline:** (1) read the Task 1 AT file → (2) build a detailed code-generation prompt → (3) send everything to Ollama in a single call → (4) clean the raw output (strip markdown fences / chatter) → (5) save to `src/traffic_light.py` → (6) syntax-check the result.

After this notebook, the remaining Task 2 steps are:
- Measure cyclomatic complexity with **radon** to choose two methods (CC > 3 and CC > 2, different loop types)
- Apply **category partition testing** to design optimized, non-redundant test cases
- Write and run the **pytest** suite that verifies the production code

## Setup

Same Ollama configuration as Task 1 — the model runs locally at `http://localhost:11434`.

- `ATS_FILE` points to the acceptance tests generated in Task 1 (`artifacts/acceptance_tests_output.txt`). This is the input that drives code generation.
- `CODE_FILE` is the output: `src/traffic_light.py`, the production code we'll test later.
- `SRC_DIR` is created automatically if it doesn't exist yet.

In [9]:
import requests
import time
import re
from pathlib import Path

# Ollama config
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3"

# Paths
ARTIFACTS_DIR = Path("artifacts")
SRC_DIR = Path("src")
SRC_DIR.mkdir(exist_ok=True)  # make sure src/ exists

ATS_FILE = ARTIFACTS_DIR / "acceptance_tests_output.txt"
CODE_FILE = SRC_DIR / "traffic_light.py"

print(f"Reading ATs from: {ATS_FILE}")
print(f"Will write code to: {CODE_FILE}")

Reading ATs from: artifacts\acceptance_tests_output.txt
Will write code to: src\traffic_light.py


## Step 1 — Read the acceptance tests from Task 1

We load the full AT file as a single string. This will be appended to our prompt so the model sees every AT table and can produce a class that satisfies all of them at once.

The preview confirms the file was read correctly and shows its header (timestamp, generation time, model) plus the start of the first AT.

In [10]:
with open(ATS_FILE, "r") as f:
    acceptance_tests = f.read()

print(f"Loaded {len(acceptance_tests)} characters of acceptance tests.")
print(acceptance_tests[:500] + "\n...")

Loaded 5580 characters of acceptance tests.
ACCEPTANCE TESTS (Task 1)
Generated: 2026-06-01 12:14:37
Total generation time: 49.7 seconds
Model: llama3


[1] Story: executeCycle  |  Dependability: Reliability
----------------------------------------------------------------------
| AT name: executeCycleTest | Reliability | |
| ---- | ---- | ---- |
| Input  | In automatic mode with North-South starts G
...


## Step 2 — Build the code-generation prompt

The prompt is carefully written to constrain the model's output:

- **One single class** named `TrafficLight` — not scattered functions, so shared state is consistent.
- **String colours** (`"GREEN"`, `"YELLOW"`, `"RED"`, `"FLASHING_RED"`) and two streets (`"NS"`, `"EW"`) — matching the vocabulary in the ATs.
- **Configurable durations** (green=30, yellow=4, all_red=2) — the same concrete values from the improved user stories.
- **Named methods** that map to the ATs (e.g. `execute_cycle`, `yellow_transition`, `manual_override`).
- **The safety rule** enforced: two crossing directions must never both be GREEN; if detected, fail-safe to FLASHING_RED.
- **Real control flow** (`if/else`, `for`, `while`) — we need non-trivial cyclomatic complexity so we can later choose methods with CC > 3 and CC > 2 for the category partition testing step.
- **Code only** — no explanations, no markdown fences. This makes the cleaning step simpler.

In [11]:
CODE_PROMPT = """You are an expert Python developer.

Below are acceptance tests for a Traffic Light System.
Generate ONE single Python class named `TrafficLight` that implements the
behavior described by these acceptance tests.

Requirements:
- Use colors as strings: "GREEN", "YELLOW", "RED", "FLASHING_RED".
- Two streets: "NS" (North-South) and "EW" (East-West).
- Configurable durations: green=30, yellow=4, all_red=2.
- Include methods that match the acceptance tests, for example:
  execute_cycle, yellow_transition, red_clearance_phase,
  check_synchronization, emergency_preemption, manual_override,
  power_failure_recovery, apply_time_based_schedule.
- Enforce the SAFETY RULE: two crossing directions must never both be GREEN.
  If a bad state is detected, set both to FLASHING_RED (fail-safe).
- Use real control flow (if/else, for loops, while loops) so the logic is testable.
- Raise ValueError (or return False) for invalid/rejected operations.

IMPORTANT OUTPUT RULES:
- Output ONLY valid Python code for the class.
- No explanations, no markdown fences, no text before or after.
"""

print("Prompt ready.")

Prompt ready.


## Step 3 — Ask Ollama to generate the code

We send the prompt + the full AT text to Ollama in a single call. The timeout is set higher (180s) than in Task 1 because generating a whole class takes longer than generating one AT table.

The call is timed — report the number your machine prints (ours took ~21.5 seconds).

In [12]:
def generate_code(ats_text):
    prompt = CODE_PROMPT + "\n\nAcceptance Tests:\n" + ats_text
    response = requests.post(
        OLLAMA_URL,
        json={"model": MODEL_NAME, "prompt": prompt, "stream": False},
        timeout=180
    )
    response.raise_for_status()
    return response.json()["response"]

print("Generating production code (this can take a minute)...\n")
start = time.time()
raw_code = generate_code(acceptance_tests)
elapsed = time.time() - start
print(f"Done in {elapsed:.1f} seconds.")
print(f"Received {len(raw_code)} characters.")

Generating production code (this can take a minute)...

Done in 21.5 seconds.
Received 4319 characters.


## Step 4 — Clean the raw output

Small local models often wrap code in `` ```python `` fences or add a sentence of explanation before/after the code. The `clean_code` function:

1. Strips markdown fences if present
2. Cuts any leading text before the first `class` or `import` statement

The result is pure Python that can be saved directly to a `.py` file. This becomes `traffic_light_old.py` — the unmodified AI output, kept in the repo to show exactly what the model produced before any human intervention.

In [13]:
def clean_code(text):
    # Remove ```python ... ``` fences if present
    fence = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1)
    # If there's leading chatter before 'class', cut to the first class/import
    idx_class = text.find("class ")
    idx_import = text.find("import ")
    starts = [i for i in (idx_class, idx_import) if i != -1]
    if starts:
        text = text[min(starts):]
    return text.strip() + "\n"

code = clean_code(raw_code)
print(code)

class TrafficLight:
    def __init__(self):
        self.colors = {"GREEN": 30, "YELLOW": 4, "RED": 2}
        self.directions = ["NS", "EW"]
        self.current_directions = {"NS": "RED", "EW": "RED"}
        self.manual_mode = False
        self.emergency_request = None

    def execute_cycle(self):
        if self.manual_mode:
            return self.apply_manual_override()

        if not self.is_safe_state():
            self.set_all_red()
            return

        for direction in self.directions:
            if self.current_directions[direction] == "GREEN":
                if self.check_synchronization(direction):
                    self.yellow_transition(direction)
                else:
                    self.set_all_red()
                    return
            elif self.current_directions[direction] == "YELLOW":
                self.red_clearance_phase(direction)

    def yellow_transition(self, direction):
        if self.current_directions[direction] == "GREEN":
      

## Step 5 — Save the production code

The cleaned code is written to `src/traffic_light.py`. This is the production code that the rest of Task 2 (and Tasks 3–4) will test.

In [14]:
with open(CODE_FILE, "w") as f:
    f.write(code)

print(f"Saved production code to: {CODE_FILE}")
print("\nNext: open src/traffic_light.py and check it looks like valid Python.")

Saved production code to: src\traffic_light.py

Next: open src/traffic_light.py and check it looks like valid Python.


## Step 6 — Syntax check

We compile the file with `py_compile` to catch obvious syntax errors before moving on. A passing check means the code is valid Python — it doesn't guarantee correctness (that's what the tests are for), but it confirms we can import it.

**Manual cleanup (from `traffic_light_old.py` → `traffic_light.py`):**

After generation, we reviewed the code and made the following hand-fixes to produce the final `traffic_light.py`:

1. **Added missing `import time`** — the model used `time.sleep()` inside methods but forgot the import. A one-line fix, typical for small local models.
2. **Refactored `apply_manual_override`** — the generated version called Python's `input()` to read commands from the keyboard, which would block forever in automated tests (`pytest` would hang). We refactored it to accept a `commands` list as a parameter instead. This keeps the control flow (the `while` loop, nested `for`, all branches) and cyclomatic complexity intact while making the method unit-testable. Removing I/O from logic is standard testing practice.

Both versions are in the repo: `traffic_light_old.py` (raw AI output) and `traffic_light.py` (cleaned, testable version).

**Next steps (outside this notebook):**
1. Run `radon cc traffic_light.py -s -a` to measure cyclomatic complexity of every method
2. Choose two methods with CC > 3 and CC > 2 using different loop types
3. Apply category partition testing to design non-redundant test cases
4. Write the pytest suite and confirm all tests pass

In [15]:
import py_compile
try:
    py_compile.compile(str(CODE_FILE), doraise=True)
    print("✓ The generated file is syntactically valid Python.")
except py_compile.PyCompileError as e:
    print("✗ Syntax error in generated code:")
    print(e)
    print("\nThat's OK - we'll fix it in the next step.")

✓ The generated file is syntactically valid Python.
